# nb_LeerYAML_Config

Lee el archivo de configuración `Sales_AW.yml` desde el Lakehouse,
extrae las entidades definidas y retorna un array JSON al pipeline.

## Entrada
| Campo | Valor |
|-------|-------|
| Archivo | `Files/config/Sales_AW.yml` |
| Lakehouse | `lh_Sales_AW` (default lakehouse del notebook) |

## Salida
Array JSON retornado via `notebookutils.notebook.exit()`:
```json
[
  { "query": "SELECT ... FROM SalesLT.Customer", "TablaDestino": "SalesLT_Customer" },
  { "query": "SELECT ... FROM SalesLT.Product",  "TablaDestino": "SalesLT_Product"  }
]
```

## Cómo lo usa el pipeline
El pipeline `pl_CargaDinamica_yaml` captura el output con:
```
@json(activity('Notebook_LeerYAML').output.exitValue)
```
Y lo asigna a `varConfig` (Array) via `SetVariable`.
El `ForEach` itera: `@item().query` → source · `@item().TablaDestino` → sink.

In [1]:
config_path = "/lakehouse/default/Files/config/Sales_AW.yml"
#destination_schema = config_path.split("/")[-1].replace(".yml", "")  # ← nueva

#print(f"destination schema    : {destination_schema}")

StatementMeta(, 94965d55-0253-4dd2-ba5b-75120df52a2e, 3, Finished, Available, Finished, False)

destination schema    : Sales_AW


In [2]:
import yaml
import json

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

destination_schema = config_path.split("/")[-1].replace(".yml", "")
source_name      = config["source"]["name"]
destination_name = config["destination"]["name"]
entities         = config.get("entities", [])

print(f"Fuente    : {source_name}")
print(f"Destino   : {destination_name}")
print(f"Esquema   : {destination_schema}")   # ← nueva
print(f"Entidades : {len(entities)}")

StatementMeta(, 94965d55-0253-4dd2-ba5b-75120df52a2e, 4, Finished, Available, Finished, False)

Fuente    : sql_Sales_AW
Destino   : lh_Sales_AW
Esquema   : Sales_AW
Entidades : 5


In [3]:
#Crear el esquema si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {destination_schema}")
print(f"Esquema '{destination_schema}' listo")

StatementMeta(, 237689b1-12ff-48fc-b22c-83fa5b4bdb15, 5, Finished, Available, Finished, False)

Esquema 'Sales_AW' listo


In [4]:
pipeline_array = [
    {
        "query":        entity.get("query", entity.get("sp", "")).strip(),
        "TablaDestino": entity["destination_table"]
    }
    for entity in entities
]

print("Entidades a procesar:")
for item in pipeline_array:
    primera_linea = item["query"].splitlines()[0] if item["query"] else "(sin query)"
    print(f"  {item['TablaDestino']:<35} <- {primera_linea[:55]}")

StatementMeta(, 237689b1-12ff-48fc-b22c-83fa5b4bdb15, 6, Finished, Available, Finished, False)

Entidades a procesar:
  SalesLT_ProductCategory             <- SELECT ProductCategoryID, ParentProductCategoryID, Name
  SalesLT_ProductModel                <- SELECT ProductModelID, Name, ModifiedDate
  SalesLT_Product                     <- SELECT ProductID, Name, ProductNumber, Color, StandardC
  SalesLT_SalesOrderHeader            <- SELECT SalesOrderID, OrderDate, DueDate, ShipDate, Stat
  SalesLT_SalesOrderDetail            <- SELECT SalesOrderID, SalesOrderDetailID, OrderQty, Prod


In [5]:
result = json.dumps({
      "destination_schema": destination_schema,
      "entities":           pipeline_array
  })
notebookutils.notebook.exit(result)


StatementMeta(, 237689b1-12ff-48fc-b22c-83fa5b4bdb15, 7, Finished, Available, Finished, False)

ExitValue: {"destination_schema": "Sales_AW", "entities": [{"query": "SELECT ProductCategoryID, ParentProductCategoryID, Name, ModifiedDate\nFROM SalesLT.ProductCategory\n-- WHERE ParentProductCategoryID IS NOT NULL", "TablaDestino": "SalesLT_ProductCategory"}, {"query": "SELECT ProductModelID, Name, ModifiedDate\nFROM SalesLT.ProductModel\n-- WHERE Name LIKE '%Road%'", "TablaDestino": "SalesLT_ProductModel"}, {"query": "SELECT ProductID, Name, ProductNumber, Color, StandardCost, ListPrice,\n       Size, Weight, ProductCategoryID, ProductModelID,\n       SellStartDate, ModifiedDate\nFROM SalesLT.Product\n-- WHERE ListPrice > 0", "TablaDestino": "SalesLT_Product"}, {"query": "SELECT SalesOrderID, OrderDate, DueDate, ShipDate, Status,\n       CustomerID, SubTotal, TaxAmt, Freight,\n       ISNULL(SubTotal + TaxAmt + Freight, 0) AS TotalDue,\n       ModifiedDate\nFROM SalesLT.SalesOrderHeader\n-- WHERE OrderDate >= '@watermark'  -- reemplazar @watermark por fecha para carga incremental", "

In [2]:
import sempy.fabric as fabric

client = fabric.FabricRestClient()

# ID del workspace y del artefacto (SQL Database)
workspace_id = "57732b87-743f-4a56-9e20-7f1b5516f48c"
artifact_id  = "bd325e18-02a1-4cbf-a1eb-fa1eded49984"

response = client.get(
    f"v1/workspaces/{workspace_id}/items/{artifact_id}/dataSources"
)

response.json()

StatementMeta(, 94e4fdc8-fa60-4856-be1b-1a0439c07c03, 4, Finished, Available, Finished, False)

FabricHTTPException: 404 Not Found for url: https://api.fabric.microsoft.com/v1/workspaces/57732b87-743f-4a56-9e20-7f1b5516f48c/items/bd325e18-02a1-4cbf-a1eb-fa1eded49984/dataSources
Error: {"requestId":"2879ba36-a014-4f0d-9bc5-b2179645b4cf","errorCode":"EntityNotFound","message":"The requested resource could not be found","isRetriable":false}
Headers: {'Content-Length': '155', 'Content-Type': 'application/json; charset=utf-8', 'x-ms-public-api-error-code': 'EntityNotFound', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'Access-Control-Expose-Headers': 'RequestId', 'request-redirected': 'true', 'home-cluster-uri': 'https://wabi-paas-1-scus-redirect.analysis.windows.net/', 'RequestId': '2879ba36-a014-4f0d-9bc5-b2179645b4cf', 'Date': 'Thu, 25 Jun 2026 21:23:08 GMT'}

In [3]:
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
artifact_id  = "bd325e18-02a1-4cbf-a1eb-fa1eded49984"
client = fabric.FabricRestClient()

# Endpoint correcto para ver dependencias de un item
response = client.get(
    f"v1/workspaces/{workspace_id}/items/{artifact_id}/connections"
)
print(response.json())

StatementMeta(, 94e4fdc8-fa60-4856-be1b-1a0439c07c03, 5, Finished, Available, Finished, False)

{'value': []}
